Step 1: Copy url of the reigion and paste into "baserurl" variable           
Step 2: Write down the name of that region into "loc" variable.              
Step 3: Write the name of the output file into "finalfile" variable.          
Step 4: Upload the example file : bonbanh_used_car_sample_20.csv.              
Step 5: Run all code at once.  

In [ ]:
baseurl = "https://bonbanh.com/vinh-long/oto-cu-da-qua-su-dung"
loc = "Vĩnh Long"
finalfile = "bonbanh_VinhLong_car_dataset.csv"

First get the car link

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time


# ============================================================
# 1. URL
# ============================================================

BASE_URL = baseurl
BASE_DOMAIN = "https://bonbanh.com/"


# ============================================================
# 2. HEADER
# ============================================================

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}


# ============================================================
# 3. SESSION
# ============================================================

session = requests.Session()
session.headers.update(headers)


# ============================================================
# 4. Save car links
# ============================================================

car_links = set()


# ============================================================
# 5. Automatically crawl til the end of the pages
# ============================================================

page = 1

while True:

    # --------------------------------------------------------
    #  URL
    # --------------------------------------------------------

    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}-/page,{page}"

    print("\n==============================")
    print(f"Đang cào page {page}")
    print(url)
    print("==============================")

    try:

        response = session.get(
            url,
            timeout=20
        )

        print("Status:", response.status_code)

        # Nếu server trả lỗi
        if response.status_code != 200:
            print("HTTP Error -> Dừng.")
            break

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # ----------------------------------------------------
        # Retrieve car link
        # ----------------------------------------------------

        cars = soup.select('a[itemprop="url"]')

        print("Số link tìm được:", len(cars))

        # ----------------------------------------------------
        # If page have no car => page is ended
        # ----------------------------------------------------

        if len(cars) == 0:

            print("\nKhông còn xe.")
            print("Đã đến cuối danh sách.")

            break

        # ----------------------------------------------------
        #  LINK Car
        # ----------------------------------------------------

        old_count = len(car_links)

        for car in cars:

            href = car.get("href")

            if href:

                full_url = urljoin(
                    BASE_DOMAIN,
                    href
                )

                # Take car  link and add to the base domain "https://bonbanh.com/"
                if "bonbanh.com" in full_url:
                    car_links.add(full_url)

        new_count = len(car_links) - old_count

        print("Link mới:", new_count)
        print("Tổng link:", len(car_links))

        # ----------------------------------------------------
        # IF the number of link not increase
        # this will avoid loop
        # ----------------------------------------------------

        if new_count == 0:

            print("\nPage không có link mới.")
            print("Có thể đã đến cuối danh sách.")
            break

        # ----------------------------------------------------
        # Move to next page
        # ----------------------------------------------------

        page += 1

        # Resting time between request
        time.sleep(1)


    except requests.exceptions.RequestException as e:

        print("Lỗi request:", e)

        #Rest and retry
        time.sleep(5)

        continue


    except Exception as e:

        print("Lỗi:", e)
        break

# Move to next page
page += 1

# ============================================
# BACKUP
# ============================================

df_backup = pd.DataFrame({
    "url": sorted(car_links)
})

df_backup.to_csv(
    "car_links_backup.csv",
    index=False,
    encoding="utf-8-sig"
)

print("💾 Đã backup:", len(car_links), "links")

time.sleep(1)


# ============================================================
# 6. Result
# ============================================================

print("\n========================================")
print("ĐÃ HOÀN THÀNH")
print("========================================")

print("Số page đã cào:", page)
print("TỔNG SỐ LINK XE:", len(car_links))

print("========================================")


# ============================================================
# 7. Save into CSV
# ============================================================

df = pd.DataFrame({
    "url": sorted(car_links)
})


df.to_csv(
    "car_links.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n========================================")
print("Đã lưu file: car_links.csv")
print("========================================")


Đang cào page 1
https://bonbanh.com/vinh-long/oto-cu-da-qua-su-dung
Status: 200
Số link tìm được: 40
Link mới: 39
Tổng link: 39

Đang cào page 2
https://bonbanh.com/vinh-long/oto-cu-da-qua-su-dung-/page,2
Status: 200
Số link tìm được: 40
Link mới: 0
Tổng link: 39

Page không có link mới.
Có thể đã đến cuối danh sách.
💾 Đã backup: 39 links

ĐÃ HOÀN THÀNH
Số page đã cào: 3
TỔNG SỐ LINK XE: 39

Đã lưu file: car_links.csv


In [ ]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

# ============================================================
# 1. Read FILE CSV contain car link
# ============================================================

input_file = "/content/car_links.csv"

df_links = pd.read_csv(input_file)

# Kiểm tra cột url
if "url" not in df_links.columns:
    raise ValueError(
        f"Không tìm thấy cột 'url'. Các cột hiện có: {df_links.columns.tolist()}"
    )

# Lấy URL, bỏ ô trống
car_urls = (
    df_links["url"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

# Nếu link là relative URL thì nối với domain Bonbanh
BASE_URL = "https://bonbanh.com/"

car_urls = [
    urljoin(BASE_URL, url)
    for url in car_urls
]

# Filter out URLs that are not specific car listings (e.g., general bonbanh.com pages)
car_urls = [url for url in car_urls if 'xe-' in url]

print(f"Số lượng link xe: {len(car_urls)}")

print("\n5 link đầu tiên:")
for url in car_urls[:5]:
    print(url)


# ============================================================
# 2. REQUEST HEADER
# ============================================================

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    )
}


# ============================================================
# 3. LẤY TÊN XE (KHÔNG TÁCH GIÁ Ở ĐÂY)
# ============================================================

def get_car_name_and_price(soup):

    title = soup.select_one(".title h1")

    if not title:
        return None, None

    text = title.get_text(" ", strip=True)

    # Normalize all whitespace (including newlines and tabs) to a single space
    text = re.sub(r'\s+', ' ', text).strip()

    # Bỏ "Xe" ở đầu
    text = re.sub(
        r"^\s*Xe\s+",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    # Return the cleaned car_name with price string still included, and None for price.
    # Price extraction will be handled in a later step (WQDqk3sJF6iP).
    return text, None


# ============================================================
# 4. LẤY TOÀN BỘ THÔNG SỐ KỸ THUẬT
# ============================================================

def get_car_specs(soup):

    specs = {}

    rows = soup.select(
        "#sgg .box_car_detail .row, "
        "#sgg .box_car_detail .row_last"
    )

    for row in rows:

        label = row.select_one(".label")
        value = row.select_one(".inp")

        if label and value:

            key = label.get_text(
                " ",
                strip=True
            )

            value_text = value.get_text(
                " ",
                strip=True
            )

            # Xóa dấu :
            key = key.replace(":", "").strip()

            specs[key] = value_text

    return specs


# ============================================================
# 5. CRAWL
# ============================================================

data = []

for i, url in enumerate(car_urls, start=1):

    print("=" * 70)
    print(f"[{i}/{len(car_urls)}]")
    print(url)

    try:

        # ----------------------------------------------------
        # Request
        # ----------------------------------------------------

        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        # ----------------------------------------------------
        # BeautifulSoup
        # ----------------------------------------------------

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # ----------------------------------------------------
        # Tên + giá
        # ----------------------------------------------------

        car_name, price = get_car_name_and_price(soup)

        # ----------------------------------------------------
        # Thông số
        # ----------------------------------------------------

        specs = get_car_specs(soup)

        # ----------------------------------------------------
        # Tạo record
        # ----------------------------------------------------

        car = {

            "car_name": car_name,

            "price": price, # This will be None, to be populated by WQDqk3sJF6iP

            "year": specs.get(
                "Năm sản xuất"
            ),

            "condition": specs.get(
                "Tình trạng"
            ),

            "mileage": specs.get(
                "Số Km đã đi"
            ),

            "origin": specs.get(
                "Xuất xứ"
            ),

            "body_type": specs.get(
                "Kiểu dáng"
            ),

            "transmission": specs.get(
                "Hộp số"
            ),

            "engine": specs.get(
                "Động cơ"
            ),

            "exterior_color": specs.get(
                "Màu ngoại thất"
            ),

            "interior_color": specs.get(
                "Màu nội thất"
            ),

            "seats": specs.get(
                "Số chỗ ngồi"
            ),

            "doors": specs.get(
                "Số cửa"
            ),

            "drive": specs.get(
                "Dẫn động"
            ),

            "url": url
        }

        data.append(car)

        print("✓", car_name)
        print("  Price:", price)
        print("  Specs:", len(specs))

        # Tránh request quá nhanh
        time.sleep(1)

    except Exception as e:

        print("✗ ERROR:", e)

        # Vẫn lưu URL để biết xe nào lỗi
        data.append({

            "car_name": None,
            "price": None,
            "year": None,
            "condition": None,
            "mileage": None,
            "origin": None,
            "body_type": None,
            "transmission": None,
            "engine": None,
            "exterior_color": None,
            "interior_color": None,
            "seats": None,
            "doors": None,
            "drive": None,
            "url": url
        })


# ============================================================
# 6. DATAFRAME
# ============================================================

df = pd.DataFrame(data)


# ============================================================
# 7. SẮP XẾP CỘT
# ============================================================

columns = [
    "car_name",
    "price",
    "year",
    "condition",
    "mileage",
    "origin",
    "body_type",
    "transmission",
    "engine",
    "exterior_color",
    "interior_color",
    "seats",
    "doors",
    "drive",
    "url"
]

df = df[columns]


# ============================================================
# 8. HIỂN THỊ
# ============================================================

display(df)


# ============================================================
# 9. KIỂM TRA DATASET
# ============================================================

print("\nDataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())


# ============================================================
# 10. LƯU CSV
# ============================================================

output_file = "/content/car_dataset.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n✓ Saved as {output_file}")

Số lượng link xe: 8

5 link đầu tiên:
https://bonbanh.com/gia-xe-oto
https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2013-6938331
https://bonbanh.com/xe-ford-everest-2.5l-4x2-mt-2008-6938339
https://bonbanh.com/xe-honda-civic-1.8-at-2007-6928278
https://bonbanh.com/xe-kia-carnival-premium-2.2d-2022-6979824
[1/8]
https://bonbanh.com/gia-xe-oto
✓ None
  Price: None
  Specs: 0
[2/8]
https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2013-6938331
✓ Chevrolet Aveo 1.5 MT 2013 - 116 Triệu
  Price: None
  Specs: 12
[3/8]
https://bonbanh.com/xe-ford-everest-2.5l-4x2-mt-2008-6938339
✓ Ford Everest 2.5L 4x2 MT 2008 - 200 Triệu
  Price: None
  Specs: 12
[4/8]
https://bonbanh.com/xe-honda-civic-1.8-at-2007-6928278
✓ Honda Civic 1.8 AT 2007 - 180 Triệu
  Price: None
  Specs: 12
[5/8]
https://bonbanh.com/xe-kia-carnival-premium-2.2d-2022-6979824
✓ Kia Carnival Premium 2.2D 2022 - 920 Triệu
  Price: None
  Specs: 12
[6/8]
https://bonbanh.com/xe-mazda-2-1.5-at-2015-6933095
✓ Mazda 2 1.5 AT 2015 - 270 Triệu
  P

,car_name,price,year,condition,mileage,origin,body_type,transmission,engine,exterior_color,interior_color,seats,doors,drive,url
0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,https://bonbanh.com/gia-xe-oto
1,Chevrolet Aveo 1.5 MT 2013 - 116 Triệu,None,2013,Xe đã dùng,"140,000 Km",Lắp ráp trong nước,Sedan,Số tay,Xăng 1.5 L,Đỏ,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2...
2,Ford Everest 2.5L 4x2 MT 2008 - 200 Triệu,None,2008,Xe đã dùng,"112,000 Km",Lắp ráp trong nước,SUV,Số tay,Dầu 2.5 L,Cát,Kem,7 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-ford-everest-2.5l-4x2-m...
3,Honda Civic 1.8 AT 2007 - 180 Triệu,None,2007,Xe đã dùng,0 Km,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-honda-civic-1.8-at-2007...
4,Kia Carnival Premium 2.2D 2022 - 920 Triệu,None,2022,Xe đã dùng,"85,000 Km",Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,Vàng,8 chỗ,5 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-kia-carnival-premium-2....
5,Mazda 2 1.5 AT 2015 - 270 Triệu,None,2015,Xe đã dùng,"86,879 Km",Nhập khẩu,Sedan,Số tự động,Xăng 1.5 L,Nâu,Đen,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-mazda-2-1.5-at-2015-693...
6,Mitsubishi Attrage 1.2 MT Eco 2019 - 190 Triệu,None,2019,Xe đã dùng,"119,000 Km",Nhập khẩu,Sedan,Số tay,Xăng 1.2 L,Đỏ,Đen,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-mitsubishi-attrage-1.2-...
7,Toyota Innova G 2009 - 149 Triệu,None,2009,Xe đã dùng,"161,000 Km",Lắp ráp trong nước,Crossover,Số tay,Xăng 2.0 L,Bạc,Kem,8 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-toyota-innova-g-2009-69...



Dataset shape: (8, 15)

Missing values:
car_name          1
price             8
year              1
condition         1
mileage           1
origin            1
body_type         1
transmission      1
engine            1
exterior_color    1
interior_color    1
seats             1
doors             1
drive             1
url               0
dtype: int64

✓ Saved as /content/car_dataset.csv


In [ ]:
# ============================================================
# BACKUP NGAY LẬP TỨC DATA ĐANG CRAWL
# ============================================================

import pandas as pd
from datetime import datetime

if "data" not in globals():
    print("❌ Không tìm thấy biến data.")
else:
    # Lấy dữ liệu hiện tại
    backup_df = pd.DataFrame(data)

    # Tên file backup có thời gian
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = f"/content/backup_{timestamp}.csv"

    # Lưu
    backup_df.to_csv(
        backup_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 60)
    print("💾 BACKUP THÀNH CÔNG")
    print(f"Số xe đã crawl hiện tại: {len(backup_df)}")
    print(f"File backup: {backup_file}")
    print("=" * 60)

    display(backup_df.tail())

💾 BACKUP THÀNH CÔNG
Số xe đã crawl hiện tại: 8
File backup: /content/backup_20260913_091713.csv


,car_name,price,year,condition,mileage,origin,body_type,transmission,engine,exterior_color,interior_color,seats,doors,drive,url
3,Honda Civic 1.8 AT 2007 - 180 Triệu,None,2007,Xe đã dùng,0 Km,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-honda-civic-1.8-at-2007...
4,Kia Carnival Premium 2.2D 2022 - 920 Triệu,None,2022,Xe đã dùng,"85,000 Km",Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,Vàng,8 chỗ,5 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-kia-carnival-premium-2....
5,Mazda 2 1.5 AT 2015 - 270 Triệu,None,2015,Xe đã dùng,"86,879 Km",Nhập khẩu,Sedan,Số tự động,Xăng 1.5 L,Nâu,Đen,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-mazda-2-1.5-at-2015-693...
6,Mitsubishi Attrage 1.2 MT Eco 2019 - 190 Triệu,None,2019,Xe đã dùng,"119,000 Km",Nhập khẩu,Sedan,Số tay,Xăng 1.2 L,Đỏ,Đen,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-mitsubishi-attrage-1.2-...
7,Toyota Innova G 2009 - 149 Triệu,None,2009,Xe đã dùng,"161,000 Km",Lắp ráp trong nước,Crossover,Số tay,Xăng 2.0 L,Bạc,Kem,8 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-toyota-innova-g-2009-69...


In [ ]:
# @title Tách giá từ carname

import pandas as pd
import re


# ============================================================
# 1. ĐỌC FILE CSV
# ============================================================

input_file = "/content/car_dataset.csv"

df = pd.read_csv(
    input_file,
    encoding="utf-8-sig"
)

print("Các cột hiện có:")
print(df.columns.tolist())


# ============================================================
# 2. HÀM TÁCH GIÁ
# ============================================================

def extract_price(text):

    if pd.isna(text):
        return None

    text = str(text).lower().strip()

    # --------------------------------------------------------
    # Trường hợp: 1 tỷ 250 triệu
    # --------------------------------------------------------

    pattern_ty_trieu = r'(\d+(?:[.,]\d+)?)\s*(?:tỷ|tỉ)\s*(\d+(?:[.,]\d+)?)?\s*(?:triệu|tr)?'

    match = re.search(pattern_ty_trieu, text)

    if match:

        ty = float(
            match.group(1).replace(",", ".")
        )

        trieu = match.group(2)

        if trieu:
            trieu = float(
                trieu.replace(",", ".")
            )
        else:
            trieu = 0

        return int(
            ty * 1_000_000_000
            + trieu * 1_000_000
        )


    # --------------------------------------------------------
    # Trường hợp: 299 triệu / 299 tr
    # --------------------------------------------------------

    pattern_trieu = r'(\d+(?:[.,]\d+)?)\s*(?:triệu|tr)\b'

    match = re.search(
        pattern_trieu,
        text
    )

    if match:

        trieu = float(
            match.group(1).replace(",", ".")
        )

        return int(
            trieu * 1_000_000
        )


    # --------------------------------------------------------
    # Trường hợp: 1 tỷ / 1.25 tỷ
    # --------------------------------------------------------

    pattern_ty = r'(\d+(?:[.,]\d+)?)\s*(?:tỷ|tỉ)\b'

    match = re.search(
        pattern_ty,
        text
    )

    if match:

        ty = float(
            match.group(1).replace(",", ".")
        )

        return int(
            ty * 1_000_000_000
        )


    # Không tìm thấy giá
    return None


# ============================================================
# 3. TÁCH GIÁ TỪ carname
# ============================================================

df["price"] = df["car_name"].apply(
    extract_price
)


# ============================================================
# 4. HIỂN THỊ KẾT QUẢ
# ============================================================

print("\nKết quả:")

print(
    df[["car_name", "price"]].head(20)
)


# ============================================================
# 5. LƯU FILE MỚI
# ============================================================

output_file = "car_dataset_p2.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n==============================")
print("Đã xử lý xong!")
print("File:", output_file)
print("==============================")

Các cột hiện có:
['car_name', 'price', 'year', 'condition', 'mileage', 'origin', 'body_type', 'transmission', 'engine', 'exterior_color', 'interior_color', 'seats', 'doors', 'drive', 'url']

Kết quả:
                                         car_name        price
0                                             NaN          NaN
1          Chevrolet Aveo 1.5 MT 2013 - 116 Triệu  116000000.0
2       Ford Everest 2.5L 4x2 MT 2008 - 200 Triệu  200000000.0
3             Honda Civic 1.8 AT 2007 - 180 Triệu  180000000.0
4      Kia Carnival Premium 2.2D 2022 - 920 Triệu  920000000.0
5                 Mazda 2 1.5 AT 2015 - 270 Triệu  270000000.0
6  Mitsubishi Attrage 1.2 MT Eco 2019 - 190 Triệu  190000000.0
7                Toyota Innova G 2009 - 149 Triệu  149000000.0

Đã xử lý xong!
File: car_dataset_p2.csv


In [ ]:
import pandas as pd
# Load the CSV file
file_path = "/content/car_dataset_p2.csv"
df = pd.read_csv(file_path)

# Add the 'location' column with the value loc
df['location'] = loc

# Save the modified DataFrame back to the CSV file
df.to_csv(file_path, index=False)

# Display the first 5 rows of the modified DataFrame to verify
display(df.head())

,car_name,price,year,condition,mileage,origin,body_type,transmission,engine,exterior_color,interior_color,seats,doors,drive,url,location
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://bonbanh.com/gia-xe-oto,Vĩnh Long
1,Chevrolet Aveo 1.5 MT 2013 - 116 Triệu,116000000.0,2013.0,Xe đã dùng,"140,000 Km",Lắp ráp trong nước,Sedan,Số tay,Xăng 1.5 L,Đỏ,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2...,Vĩnh Long
2,Ford Everest 2.5L 4x2 MT 2008 - 200 Triệu,200000000.0,2008.0,Xe đã dùng,"112,000 Km",Lắp ráp trong nước,SUV,Số tay,Dầu 2.5 L,Cát,Kem,7 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-ford-everest-2.5l-4x2-m...,Vĩnh Long
3,Honda Civic 1.8 AT 2007 - 180 Triệu,180000000.0,2007.0,Xe đã dùng,0 Km,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-honda-civic-1.8-at-2007...,Vĩnh Long
4,Kia Carnival Premium 2.2D 2022 - 920 Triệu,920000000.0,2022.0,Xe đã dùng,"85,000 Km",Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,Vàng,8 chỗ,5 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-kia-carnival-premium-2....,Vĩnh Long


In [ ]:
import re

# Function to clean the car_name column by removing price strings
def clean_car_name_from_price_string(car_name_text):
    if pd.isna(car_name_text):
        return car_name_text

    # Convert to string and clean up newlines and tabs
    cleaned_text = str(car_name_text).replace('\n', ' ').replace('\t', ' ').strip()

    # Regex to find and remove price patterns at the end of the string.
    # This covers various formats like " - 95 Triệu", " - 1 Tỷ 350 Triệu", " - 1.25 Tỷ", etc.
    price_patterns_to_remove = r'\s*-\s*\d+(?:[.,]\d+)?\s*(?:triệu|tr|tỷ|tỉ)\b(?:\s*\d+(?:[.,]\d+)?\s*(?:triệu|tr)\b)?\s*$'

    # Remove the matched price pattern from the end of the string
    cleaned_text = re.sub(price_patterns_to_remove, '', cleaned_text, flags=re.IGNORECASE).strip()

    # Also remove "Xe " prefix if it exists
    cleaned_text = re.sub(r"^\s*Xe\s+", "", cleaned_text, flags=re.IGNORECASE)

    return cleaned_text

# Apply the cleaning function to the 'car_name' column
df['car_name'] = df['car_name'].apply(clean_car_name_from_price_string)

# Display the first few rows to verify the cleaned car_name
display(df.head())

,car_name,price,year,condition,mileage,origin,body_type,transmission,engine,exterior_color,interior_color,seats,doors,drive,url,location
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://bonbanh.com/gia-xe-oto,Vĩnh Long
1,Chevrolet Aveo 1.5 MT 2013,116000000.0,2013.0,Xe đã dùng,"140,000 Km",Lắp ráp trong nước,Sedan,Số tay,Xăng 1.5 L,Đỏ,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2...,Vĩnh Long
2,Ford Everest 2.5L 4x2 MT 2008,200000000.0,2008.0,Xe đã dùng,"112,000 Km",Lắp ráp trong nước,SUV,Số tay,Dầu 2.5 L,Cát,Kem,7 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-ford-everest-2.5l-4x2-m...,Vĩnh Long
3,Honda Civic 1.8 AT 2007,180000000.0,2007.0,Xe đã dùng,0 Km,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-honda-civic-1.8-at-2007...,Vĩnh Long
4,Kia Carnival Premium 2.2D 2022,920000000.0,2022.0,Xe đã dùng,"85,000 Km",Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,Vàng,8 chỗ,5 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-kia-carnival-premium-2....,Vĩnh Long


In [ ]:
def extract_brand_model(car_name):
    if pd.isna(car_name):
        return '', ''
    # Clean the car_name by replacing common separators and splitting
    cleaned_name = str(car_name).replace('\n', ' ').replace('\t', ' ').strip()
    parts = cleaned_name.split(maxsplit=1)

    brand = parts[0] if parts else ''
    model = parts[1] if len(parts) > 1 else ''

    return brand, model

def remove_year_from_model(model_text):
    if pd.isna(model_text) or not isinstance(model_text, str):
        return model_text
    # Regex to find a four-digit number (year) at the end of the string
    # Optionally followed by spaces and other characters that might be part of the year description.
    # Ensures it's a year-like pattern (e.g., ' 2012', ' 2024')
    cleaned_model = re.sub(r'\s+\d{4}\s*$', '', model_text).strip()
    return cleaned_model

# Apply the function to create 'brand' and 'model' columns
df[['brand', 'model']] = df['car_name'].apply(lambda x: pd.Series(extract_brand_model(x)))

# Apply the function to remove the year from the 'model' column
df['model'] = df['model'].apply(remove_year_from_model)

# Display the DataFrame with the new columns
display(df.head())

import pandas as pd

# Load the target file to get its column names
target_file_path = '/content/bonbanh_used_cars_sample_20 (1).csv'
df_target = pd.read_csv(target_file_path)
target_columns = df_target.columns.tolist()

# Use the 'df' DataFrame from the kernel state (which still has all columns including original price, mileage, drive)
# Rename columns in 'df' to match target columns
df_renamed = df.rename(columns={
    'price': 'price_vnd',
    'mileage': 'mileage_km',
    'drive': 'drivetrain'
})

# Select only the columns that are common between the renamed df and the target columns
columns_to_select = [col for col in target_columns if col in df_renamed.columns]
df_final = df_renamed[columns_to_select]

# Add any missing columns from target_columns (i.e., those that were in target_columns but not in df_renamed)
missing_cols = [col for col in target_columns if col not in df_final.columns]
for col in missing_cols:
    df_final[col] = pd.NA # Use pd.NA for missing values (or None, depending on desired type)

# Reorder columns to match the target_columns exactly
df_final = df_final[target_columns]

# Remove rows where all values are pd.NA
df_final = df_final.dropna(how='all')

# Save the modified DataFrame back to a new CSV file path
output_file_path = finalfile
df_final.to_csv(output_file_path, index=False)

# Display the first 5 rows of the modified DataFrame to verify
display(df_final.head())

,car_name,price,year,condition,mileage,origin,body_type,transmission,engine,exterior_color,interior_color,seats,doors,drive,url,location,brand,model
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://bonbanh.com/gia-xe-oto,Vĩnh Long,,
1,Chevrolet Aveo 1.5 MT 2013,116000000.0,2013.0,Xe đã dùng,"140,000 Km",Lắp ráp trong nước,Sedan,Số tay,Xăng 1.5 L,Đỏ,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-chevrolet-aveo-1.5-mt-2...,Vĩnh Long,Chevrolet,Aveo 1.5 MT
2,Ford Everest 2.5L 4x2 MT 2008,200000000.0,2008.0,Xe đã dùng,"112,000 Km",Lắp ráp trong nước,SUV,Số tay,Dầu 2.5 L,Cát,Kem,7 chỗ,5 cửa,RFD - Dẫn động cầu sau,https://bonbanh.com/xe-ford-everest-2.5l-4x2-m...,Vĩnh Long,Ford,Everest 2.5L 4x2 MT
3,Honda Civic 1.8 AT 2007,180000000.0,2007.0,Xe đã dùng,0 Km,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,Kem,5 chỗ,4 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-honda-civic-1.8-at-2007...,Vĩnh Long,Honda,Civic 1.8 AT
4,Kia Carnival Premium 2.2D 2022,920000000.0,2022.0,Xe đã dùng,"85,000 Km",Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,Vàng,8 chỗ,5 cửa,FWD - Dẫn động cầu trước,https://bonbanh.com/xe-kia-carnival-premium-2....,Vĩnh Long,Kia,Carnival Premium 2.2D


,brand,model,year,price_vnd,mileage_km,condition,origin,body_type,transmission,engine,exterior_color,seats,drivetrain,location
0,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vĩnh Long
1,Chevrolet,Aveo 1.5 MT,2013.0,116000000.0,"140,000 Km",Xe đã dùng,Lắp ráp trong nước,Sedan,Số tay,Xăng 1.5 L,Đỏ,5 chỗ,FWD - Dẫn động cầu trước,Vĩnh Long
2,Ford,Everest 2.5L 4x2 MT,2008.0,200000000.0,"112,000 Km",Xe đã dùng,Lắp ráp trong nước,SUV,Số tay,Dầu 2.5 L,Cát,7 chỗ,RFD - Dẫn động cầu sau,Vĩnh Long
3,Honda,Civic 1.8 AT,2007.0,180000000.0,0 Km,Xe đã dùng,Lắp ráp trong nước,Sedan,Số tự động,Xăng 1.8 L,Đen,5 chỗ,FWD - Dẫn động cầu trước,Vĩnh Long
4,Kia,Carnival Premium 2.2D,2022.0,920000000.0,"85,000 Km",Xe đã dùng,Lắp ráp trong nước,Van/Minivan,Số tự động,Dầu 2.2 L,Ghi,8 chỗ,FWD - Dẫn động cầu trước,Vĩnh Long
